# Introducción a NumPy y Pandas para Ciencia de Datos

**Ayudantía 1 · Introducción a la Ciencia de Datos (IDS)**

Este notebook recorre, de punta a punta, un flujo de trabajo típico de análisis de datos: entender por qué NumPy es la base del ecosistema, cargar un dataset real de ventas con Pandas, explorarlo, limpiarlo, consultarlo y resumirlo.

---

### Objetivos de aprendizaje

Al terminar serás capaz de:

1. Explicar qué es un *array* de NumPy y por qué la **vectorización** es más rápida que un ciclo `for`.
2. Cargar y explorar un `DataFrame` (`head`, `info`, `describe`, `isna`, `value_counts`).
3. Detectar y tratar **valores nulos** e **inválidos**, justificando la estrategia elegida.
4. Filtrar datos con condiciones simples, combinadas y por fecha.
5. Combinar NumPy y Pandas con `np.where` y `np.select`.
6. Resumir información con `groupby` y construir rankings (Top-N).

### Contenido

| # | Sección |
|---|---------|
| 1 | [NumPy: la base de todo](#1.-NumPy:-la-base-de-todo) |
| 2 | [Carga y exploración de datos](#2.-Carga-y-exploración-de-datos) |
| 3 | [Limpieza de datos](#3.-Limpieza-de-datos) |
| 4 | [Filtros y consultas](#4.-Filtros-y-consultas) |
| 5 | [NumPy y Pandas trabajando juntos](#5.-NumPy-y-Pandas-trabajando-juntos) |
| 6 | [Agregación y resumen](#6.-Agregación-y-resumen) |
| 7 | [Eliminación de columnas](#7.-Eliminación-de-columnas) |
| 8 | [Resumen y ejercicios propuestos](#8.-Resumen-y-ejercicios-propuestos) |

### Requisitos

- Python 3.10 o superior, con `numpy` y `pandas` instalados.
- El archivo `ventas_gigantes1.csv` en la misma carpeta que este notebook. Se genera ejecutando antes el notebook `dataset ayudantia 1.ipynb`.

## Configuración del entorno

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

# Presentación de resultados: dos decimales y separador de miles
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)

# Ruta del dataset (generado por 'dataset ayudantia 1.ipynb')
RUTA_CSV = Path("ventas_gigantes1.csv")

print(f"NumPy  {np.__version__}")
print(f"Pandas {pd.__version__}")

NumPy  2.3.5
Pandas 2.3.3


<a id="1.-NumPy:-la-base-de-todo"></a>
## 1. NumPy: la base de todo

Antes de trabajar con Pandas conviene entender qué es un *array* de NumPy: Pandas está construido sobre NumPy, y casi todo lo que hace rápido a Pandas viene de ahí.

### 1.1 Lista de Python vs. array de NumPy

In [2]:
ventas_lista = [100, 200, 300, 400]      # Lista de Python
ventas_array = np.array(ventas_lista)    # Array de NumPy

print("Lista:", ventas_lista)
print("Array:", ventas_array)
print()
print("Tipo de la lista:", type(ventas_lista).__name__)
print("Tipo del array:  ", type(ventas_array).__name__)
print("dtype del array: ", ventas_array.dtype)
print("Forma del array: ", ventas_array.shape)

Lista: [100, 200, 300, 400]
Array: [100 200 300 400]

Tipo de la lista: list
Tipo del array:   ndarray
dtype del array:  int64
Forma del array:  (4,)


Al imprimirlos se nota una diferencia estética (la lista lleva comas, el array no), pero lo importante es otra: **NumPy guarda los datos de forma contigua en memoria y con un único tipo de dato**. Eso lo hace mucho más rápido para cálculos masivos.

### 1.2 Estadísticas básicas

In [3]:
print("Media:              ", np.mean(ventas_array))
print("Desviación estándar:", np.std(ventas_array))
print("Mediana:            ", np.median(ventas_array))
print("Percentiles (10, 25, 50, 75, 90):", np.percentile(ventas_array, [10, 25, 50, 75, 90]))

Media:               250.0
Desviación estándar: 111.80339887498948
Mediana:             250.0
Percentiles (10, 25, 50, 75, 90): [130. 175. 250. 325. 370.]


### 1.3 Vectorización: operar sobre todos los elementos a la vez

Ejemplo: agregar el IVA (19 %) a todas las ventas. Primero con el enfoque tradicional (lista y ciclo `for`) y luego con NumPy.

In [4]:
# Enfoque tradicional: lista vacía + recorrer + multiplicar + guardar
ventas_iva_lista = []
for v in ventas_lista:
    ventas_iva_lista.append(v * 1.19)

print("Con lista y for:", ventas_iva_lista)

Con lista y for: [119.0, 238.0, 357.0, 476.0]


In [5]:
# Enfoque vectorizado: una sola expresión
ventas_iva = ventas_array * 1.19

print("Con NumPy:      ", ventas_iva)

Con NumPy:       [119. 238. 357. 476.]


| Enfoque | Qué hay que escribir | Cómo se ejecuta |
|---|---|---|
| **Ciclo `for`** | Crear lista vacía, recorrer, multiplicar y guardar | Elemento por elemento, en el intérprete de Python |
| **Vectorización** | `ventas_array * 1.19` | Toda la operación en código compilado, sobre bloques de memoria |

Con 4 datos la diferencia es invisible, pero crece con el volumen de datos y con la complejidad de la operación. Midámoslo con un millón de elementos (los tiempos exactos dependen de tu equipo):

In [6]:
N = 1_000_000
lista_grande = list(range(N))
array_grande = np.arange(N)

inicio = perf_counter()
_ = [v * 1.19 for v in lista_grande]
t_lista = perf_counter() - inicio

inicio = perf_counter()
_ = array_grande * 1.19
t_numpy = perf_counter() - inicio

print(f"Lista + ciclo:  {t_lista * 1000:8.1f} ms")
print(f"NumPy vectorial:{t_numpy * 1000:8.1f} ms")
print(f"NumPy fue ~{t_lista / t_numpy:.0f} veces más rápido con {N:,} elementos")

Lista + ciclo:     140.4 ms
NumPy vectorial:    38.1 ms
NumPy fue ~4 veces más rápido con 1,000,000 elementos


### 1.4 ¿Por qué es tan rápido?

> **Idea clave.** La vectorización permite que la CPU ejecute las operaciones con instrucciones **SIMD** (*Single Instruction, Multiple Data*): el procesador recibe una sola orden (por ejemplo, «súmale 1 a todo este bloque de memoria») y la aplica a muchos datos de una sola vez.

Esto es posible porque un array de NumPy es un **bloque de memoria contiguo y homogéneo** (todos sus elementos tienen el mismo tipo), y no una lista de objetos dispersos que Python deba recorrer uno a uno. Si se mezclan tipos, esa ventaja se pierde:

In [7]:
homogeneo = np.array([1, 2, 3])
mezclado = np.array([1, "dos", 3.0])

print("Homogéneo:", homogeneo, "-> dtype", homogeneo.dtype)
print("Mezclado: ", mezclado, "-> dtype", mezclado.dtype)

Homogéneo: [1 2 3] -> dtype int64
Mezclado:  ['1' 'dos' '3.0'] -> dtype <U32


Al mezclar números y texto, NumPy convierte *todo* a texto (`<U32`) y ya no se puede operar numéricamente. Por eso en ciencia de datos cada columna debe tener un tipo consistente.

<a id="2.-Carga-y-exploración-de-datos"></a>
## 2. Carga y exploración de datos

### 2.1 Carga del dataset

Trabajaremos con 100.000 registros de ventas de una cadena de tiendas, con las columnas `fecha`, `venta`, `id_tienda`, `ciudad` y `categoria_producto`.

In [8]:
if not RUTA_CSV.exists():
    raise FileNotFoundError(
        f"No se encontró '{RUTA_CSV}'. Ejecuta primero el notebook 'dataset ayudantia 1.ipynb'."
    )

df = pd.read_csv(RUTA_CSV)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

Dataset cargado: 100,000 filas x 5 columnas


,fecha,venta,id_tienda,ciudad,categoria_producto
0,2025-05-16,358.77,48,Roma,Juguetes
1,2019-12-26,436.28,38,Madrid,Juguetes
2,2023-12-06,217.23,41,Paris,Electrónica
3,2015-05-02,204.29,29,Madrid,Juguetes
4,2016-07-16,162.73,12,Barcelona,Ropa


### 2.2 Estructura y tipos de dato

`df.info()` muestra el tipo de cada columna y cuántos valores **no nulos** tiene.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   fecha               100000 non-null  object 
 1   venta               99502 non-null   float64
 2   id_tienda           100000 non-null  int64  
 3   ciudad              100000 non-null  object 
 4   categoria_producto  100000 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 3.8+ MB


La columna `venta` tiene menos valores no nulos que las demás: **faltan registros**. Además, `fecha` aparece como `object` (texto), no como fecha; lo corregiremos en la sección de limpieza.

### 2.3 Resumen estadístico con `describe()`

In [10]:
df.describe()

,venta,id_tienda
count,"99,502.00","100,000.00"
mean,255.03,24.96
std,141.74,14.12
min,10.02,1.00
25%,132.24,13.00
50%,255.26,25.00
75%,378.11,37.00
max,500.00,49.00


> **Buena práctica.** Ejecuta `df.describe()` al inicio de todo análisis: es una radiografía instantánea de tus números y permite saber rápidamente si los datos tienen sentido.

Por qué `.describe()` es tan poderoso:

**1. Detección de errores (outliers).** Es la forma más rápida de encontrar datos imposibles.
- En una columna «Edad», un mínimo de -5 o un máximo de 350 indica que el dataset está sucio.
- En ventas, un máximo ridículamente alto frente al promedio puede ser una venta errónea o un cliente mayorista que distorsiona las estadísticas.

**2. Dispersión (`std`).** La desviación estándar indica qué tan representativo es el promedio.
- Promedio 100 y `std` 2: todas las ventas son casi iguales.
- Promedio 100 y `std` 150: hay ventas muy pequeñas y muy grandes mezcladas, y el promedio dice poco (es solo un ejemplo).

**3. Distribución (cuartiles).** Los valores 25 %, 50 % (mediana) y 75 % indican dónde se concentra la mayoría de los datos.
- Si el promedio es mucho mayor que la mediana, unos pocos valores «gigantes» están arrastrando el promedio hacia arriba.
- El rango intercuartil (75 % − 25 %) es el rango donde se mueve el 50 % central de los datos.

### 2.4 Valores nulos

In [11]:
nulos = df.isna().sum()
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": nulos / len(df) * 100,
})
resumen_nulos

,nulos,porcentaje
fecha,0,0.00
venta,498,0.50
id_tienda,0,0.00
ciudad,0,0.00
categoria_producto,0,0.00


`isna().sum()` entrega el número exacto de celdas vacías por columna. Aquí todos los nulos están en `venta`: faltan 498 de 100.000 registros (≈ 0,5 %).

### 2.5 Percentiles con NumPy y la trampa de los `NaN`

Podemos extraer una columna como array de NumPy y calcular percentiles personalizados. Pero ojo: **los `NaN` contaminan el resultado** de las funciones estadísticas normales.

In [12]:
ventas_array = df["venta"].values

# Con np.percentile, un solo NaN contagia a todo el resultado
print("np.percentile:   ", np.percentile(ventas_array, [10, 25, 50, 75, 90]))

# Con np.nanpercentile, los NaN se ignoran
print("np.nanpercentile:", np.nanpercentile(ventas_array, [10, 25, 50, 75, 90]))

np.percentile:    [nan nan nan nan nan]
np.nanpercentile: [ 58.86741036 132.23895737 255.26476732 378.10705365 451.78860362]


> **Regla práctica.** Si hay `NaN` en tus datos, usa las variantes `np.nan*` (`np.nanmean`, `np.nanstd`, `np.nanpercentile`...) o limpia los nulos antes. Pandas, en cambio, ignora los `NaN` por defecto (`df["venta"].mean()` funciona sin problemas).

### 2.6 Conteos y valores únicos

`value_counts()` indica cuántas veces aparece cada categoría; sirve, por ejemplo, para detectar tiendas con pocas ventas registradas.

In [13]:
df["id_tienda"].value_counts().head(10)

id_tienda
17    2141
14    2106
2     2098
12    2098
26    2096
13    2094
25    2087
42    2086
39    2083
28    2083
Name: count, dtype: int64

Para saber **qué tiendas tienen ventas nulas** encadenamos tres pasos:

1. Filtrar las filas con `venta` nula.
2. Seleccionar solo la columna `id_tienda`.
3. Usar `.unique()` para no ver la misma tienda repetida cientos de veces.

In [14]:
tiendas_con_problemas = df[df["venta"].isna()]["id_tienda"].unique()

print(f"Tiendas con ventas nulas: {len(tiendas_con_problemas)} de {df['id_tienda'].nunique()}")
print(np.sort(tiendas_con_problemas))

Tiendas con ventas nulas: 49 de 49
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49]


<a id="3.-Limpieza-de-datos"></a>
## 3. Limpieza de datos

La limpieza ocurre **antes** de cualquier análisis. Tres tareas: corregir tipos, tratar valores inválidos y decidir qué hacer con los nulos.

### 3.1 Corrección de tipos: la columna `fecha`

`pd.to_datetime` transforma la columna `fecha` (inicialmente un texto leído desde el CSV) en un tipo de fecha que Pandas puede manipular. Gracias a eso disponemos del accesor `.dt`:

| Expresión | Resultado |
|---|---|
| `df["fecha"].dt.year` | Año |
| `df["fecha"].dt.month` | Mes |
| `df["fecha"].dt.day_name()` | Nombre del día (`"Monday"`, `"Tuesday"`...) |

In [15]:
df["fecha"] = pd.to_datetime(df["fecha"])

df[["fecha"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   fecha   100000 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 781.4 KB


### 3.2 Valores inválidos

No todos los problemas son nulos: también hay valores que existen pero **no tienen sentido** (ventas negativas o sospechosamente bajas). La estrategia es convertirlos en `NaN` para tratarlos junto con el resto de los nulos.

Aquí consideramos inválida toda venta negativa o menor a 10. En este dataset la cuenta dará 0, pero esta verificación debe hacerse siempre antes de calcular promedios.

> **`np.where(condición, valor_si_verdadero, valor_si_falso)`** es un «If-Else» ultrarrápido que actúa sobre toda la columna a la vez.

In [16]:
invalidas = (df["venta"] < 10)   # incluye las negativas
print(f"Ventas inválidas detectadas: {invalidas.sum():,}")

df["venta"] = np.where(invalidas, np.nan, df["venta"])

print(f"Nulos en 'venta' tras marcar inválidos: {df['venta'].isna().sum():,}")

Ventas inválidas detectadas: 0
Nulos en 'venta' tras marcar inválidos: 498


### 3.3 Tratamiento de nulos: ¿eliminar o rellenar?

Hay dos estrategias habituales. Comparemos cuánta información se pierde con cada una.

**Opción A: eliminar las filas con nulos** (`dropna`).

In [17]:
df_limpio = df.dropna().reset_index(drop=True)

print(f"Filas originales:    {len(df):,}")
print(f"Filas tras dropna(): {len(df_limpio):,}")
print(f"Filas perdidas:      {len(df) - len(df_limpio):,}")

Filas originales:    100,000
Filas tras dropna(): 99,502
Filas perdidas:      498


> **Sobre `reset_index(drop=True)`.** Al eliminar filas, el índice queda con «huecos». `reset_index` lo renumera desde 0 y `drop=True` evita que el índice antiguo se agregue como una columna nueva.
>
> Si decides trabajar con `df_limpio`, todas las celdas siguientes deben usar `df_limpio` en lugar de `df`.

**Opción B: rellenar los nulos con el promedio** (`fillna`).

In [18]:
promedio_venta = df["venta"].mean()
print(f"Promedio usado para imputar: {promedio_venta:,.2f}")

df["venta"] = df["venta"].fillna(promedio_venta)

print(f"Nulos restantes en 'venta': {df['venta'].isna().sum()}")

Promedio usado para imputar: 255.03
Nulos restantes en 'venta': 0


**¿Cuál elegir?** En la mayoría de los casos eliminar directamente los nulos supone **perder información** (se descarta la fila completa, incluidas las demás columnas que sí tenían datos). Por eso suele ser preferible rellenar con el promedio, la mediana o métodos más avanzados.

Con 0,5 % de nulos la diferencia es pequeña, pero con porcentajes mayores eliminar puede sesgar el análisis. **A partir de aquí seguiremos con la Opción B**: `df` ya no tiene nulos y se puede usar con normalidad.

> **Ojo con la imputación por promedio.** Rellenar con un único valor reduce artificialmente la variabilidad de los datos. Es una solución práctica para este ejercicio, pero no siempre la más adecuada.

<a id="4.-Filtros-y-consultas"></a>
## 4. Filtros y consultas

Filtrar consiste en indexar el `DataFrame` con una **máscara booleana**: `df[condición]` devuelve solo las filas donde la condición es `True`.

### 4.1 Filtro por fecha y cálculo en una sola línea

In [19]:
promedio_desde_2021 = df[df["fecha"].dt.year >= 2021]["venta"].mean()

print(f"Venta promedio desde 2021: {promedio_desde_2021:,.2f}")

Venta promedio desde 2021: 254.88


### 4.2 Condiciones múltiples

Para combinar condiciones se usan `&` (y), `|` (o) y `~` (no). **Cada condición debe ir entre paréntesis**, porque `&` tiene mayor precedencia que `>` o `==`.

| Intención | Código |
|---|---|
| Solo diciembre | `df[df["fecha"].dt.month == 12]` |
| Ventas mayores a 400 | `df[df["venta"] > 400]` |
| Varias condiciones | `df[(df["venta"] > 400) & (df["id_tienda"] == 1)]` |
| Pertenece a una lista | `df[df["ciudad"].isin(["Madrid", "Roma"])]` |

> **Error común.** `df[df["venta"] > 100 & df["id_tienda"] == 5]` lanza un error. Lo correcto es `df[(df["venta"] > 100) & (df["id_tienda"] == 5)]`.

In [20]:
consulta = df[
    (df["venta"] > 400)
    & (df["ciudad"].isin(["Madrid", "Roma"]))
    & (df["fecha"].dt.year == 2023)
]

print(f"Ventas de Madrid o Roma, mayores a 400, durante 2023: {len(consulta):,}")
consulta.head()

Ventas de Madrid o Roma, mayores a 400, durante 2023: 707


,fecha,venta,id_tienda,ciudad,categoria_producto
205,2023-07-12,475.64,45,Roma,Electrónica
305,2023-09-24,473.16,44,Roma,Juguetes
376,2023-01-03,490.98,31,Roma,Ropa
422,2023-04-12,414.28,15,Madrid,Alimentos
539,2023-12-26,425.53,29,Roma,Ropa


<a id="5.-NumPy-y-Pandas-trabajando-juntos"></a>
## 5. NumPy y Pandas trabajando juntos

Pandas usa NumPy por debajo, así que las funciones de NumPy se pueden aplicar directamente sobre columnas de un `DataFrame`.

### 5.1 Clasificación binaria con `np.where`

Etiquetamos cada venta como `'Alto'` si es mayor o igual al promedio y `'Bajo'` en caso contrario.

In [21]:
promedio = df["venta"].mean()

df["rendimiento"] = np.where(df["venta"] >= promedio, "Alto", "Bajo")

df["rendimiento"].value_counts()

rendimiento
Alto    50291
Bajo    49709
Name: count, dtype: int64

### 5.2 Clasificación en varios tramos con `np.select`

Cuando hay más de dos categorías, `np.select` actúa como un «Switch / Case» avanzado:

| Parámetro | Descripción |
|---|---|
| `condlist` | Lista de condiciones (máscaras booleanas), evaluadas en orden. |
| `choicelist` | Valores a asignar; si se cumple la condición *i*, se usa el valor *i*. |
| `default` | Valor cuando ninguna condición se cumple (el `else` final). Si se omite, es `0`. |

Dividimos las ventas en tres tramos usando los cuantiles 33 % y 66 %:

In [22]:
umbral_bajo = df["venta"].quantile(0.33)
umbral_alto = df["venta"].quantile(0.66)

condiciones = [
    df["venta"] < umbral_bajo,
    (df["venta"] >= umbral_bajo) & (df["venta"] < umbral_alto),
    df["venta"] >= umbral_alto,
]
opciones = ["Bajo", "Medio", "Alto"]

df["tramo_venta"] = np.select(condiciones, opciones, default="Desconocido")

print(f"Umbrales: bajo < {umbral_bajo:,.2f} <= medio < {umbral_alto:,.2f} <= alto")
df["tramo_venta"].value_counts()

Umbrales: bajo < 172.02 <= medio < 332.25 <= alto


tramo_venta
Alto     34000
Medio    33000
Bajo     33000
Name: count, dtype: int64

In [23]:
df.head()

,fecha,venta,id_tienda,ciudad,categoria_producto,rendimiento,tramo_venta
0,2025-05-16,358.77,48,Roma,Juguetes,Alto,Alto
1,2019-12-26,436.28,38,Madrid,Juguetes,Alto,Alto
2,2023-12-06,217.23,41,Paris,Electrónica,Bajo,Medio
3,2015-05-02,204.29,29,Madrid,Juguetes,Bajo,Medio
4,2016-07-16,162.73,12,Barcelona,Ropa,Bajo,Bajo


<a id="6.-Agregación-y-resumen"></a>
## 6. Agregación y resumen

`groupby` aplica el patrón **dividir → aplicar → combinar**: separa las filas en grupos, calcula algo por grupo y junta los resultados.

```python
df.groupby("columna_grupo")["columna_calculo"].funcion_agregacion()
```

### 6.1 Top 5 de tiendas con mayores ventas en 2023

In [24]:
top_tiendas_2023 = (
    df[df["fecha"].dt.year == 2023]
    .groupby("id_tienda")["venta"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("--- TOP 5 TIENDAS 2023 ---")
print(top_tiendas_2023)

--- TOP 5 TIENDAS 2023 ---
id_tienda
37   53,089.56
28   52,423.26
12   51,722.18
39   50,387.42
31   49,855.30
Name: venta, dtype: float64


Desglose de la consulta, paso a paso:

| Paso | Código | Qué hace |
|---|---|---|
| Filtro | `df[df["fecha"].dt.year == 2023]` | Se queda solo con las filas de 2023. |
| Agrupación | `.groupby("id_tienda")` | Junta las filas de cada tienda en «bolsas» separadas. |
| Agregación | `["venta"].sum()` | En cada bolsa, suma la columna de ventas. |
| Orden | `.sort_values(ascending=False)` | Deja los totales más grandes arriba. |
| Límite | `.head(5)` | Conserva solo los 5 mejores. |

Para presentar los resultados podemos redondearlos con `np.round`:

In [25]:
print("--- TOP 5 TIENDAS 2023 (redondeado) ---")
print(np.round(top_tiendas_2023, 2))

--- TOP 5 TIENDAS 2023 (redondeado) ---
id_tienda
37   53,089.56
28   52,423.26
12   51,722.18
39   50,387.42
31   49,855.30
Name: venta, dtype: float64


### 6.2 Top 3 de ciudades con mayor venta total

In [26]:
top_ciudades = (
    df.groupby("ciudad")["venta"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("--- TOP 3 CIUDADES POR VENTA TOTAL ---")
print(np.round(top_ciudades, 2))

--- TOP 3 CIUDADES POR VENTA TOTAL ---
ciudad
Barcelona   5,141,357.17
Sevilla     5,110,545.75
Roma        5,110,520.79
Name: venta, dtype: float64


### 6.3 Top 3 de categorías de producto con mayor venta total

In [27]:
top_categorias = (
    df.groupby("categoria_producto")["venta"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("--- TOP 3 CATEGORÍAS POR VENTA TOTAL ---")
print(np.round(top_categorias, 2))

--- TOP 3 CATEGORÍAS POR VENTA TOTAL ---
categoria_producto
Alimentos     5,152,213.96
Ropa          5,115,600.18
Electrónica   5,086,453.55
Name: venta, dtype: float64


### 6.4 Varias métricas a la vez con `.agg()`

Con `.agg()` se calculan varias agregaciones en una sola pasada.

In [28]:
(
    df.groupby("ciudad")["venta"]
    .agg(["count", "sum", "mean", "median"])
    .sort_values("sum", ascending=False)
)

,count,sum,mean,median
ciudad,,,,
Barcelona,20133,"5,141,357.17",255.37,256.43
Sevilla,20055,"5,110,545.75",254.83,255.03
Roma,20000,"5,110,520.79",255.53,255.03
Madrid,19922,"5,088,626.87",255.43,255.03
Paris,19890,"5,051,555.48",253.97,253.90


<a id="7.-Eliminación-de-columnas"></a>
## 7. Eliminación de columnas

Supongamos que para un reporte no necesitamos la columna `id_tienda`. `drop` **no modifica `df`** a menos que reasignemos el resultado, así que podemos guardarlo en una variable nueva y conservar el original intacto.

In [29]:
df_reporte = df.drop(columns=["id_tienda"])

print("Columnas de df:         ", list(df.columns))
print("Columnas de df_reporte: ", list(df_reporte.columns))

Columnas de df:          ['fecha', 'venta', 'id_tienda', 'ciudad', 'categoria_producto', 'rendimiento', 'tramo_venta']
Columnas de df_reporte:  ['fecha', 'venta', 'ciudad', 'categoria_producto', 'rendimiento', 'tramo_venta']


> **Recomendación.** Si eliminas una columna sobre `df` mismo (`df = df.drop(...)`), cualquier celda posterior que la use dará error. Elimina columnas solo al final del análisis o trabaja sobre una copia, como arriba.

<a id="8.-Resumen-y-ejercicios-propuestos"></a>
## 8. Resumen y ejercicios propuestos

### Referencia rápida

| Etapa | Herramienta | Para qué sirve |
|---|---|---|
| Exploración | `df.head()`, `df.shape`, `df.info()` | Ver el contenido, el tamaño y los tipos. |
| | `df.describe()` | Resumen numérico y detección de anomalías. |
| | `df.isna().sum()` | Contar nulos por columna. |
| | `df["col"].value_counts()` / `.unique()` | Contar y listar categorías. |
| Limpieza | `pd.to_datetime()`, `.astype()` | Corregir tipos de dato. |
| | `df.dropna()` / `df.fillna(valor)` | Eliminar o imputar nulos. |
| | `df.drop_duplicates()` | Quitar filas repetidas. |
| Filtrado | `df[(cond1) & (cond2)]` | Consultas con condiciones combinadas. |
| NumPy + Pandas | `np.where`, `np.select` | Columnas derivadas a partir de condiciones. |
| Resumen | `groupby(...).sum()/.agg(...)` | Totales y métricas por grupo. |

### Ideas clave

- **Vectoriza siempre que puedas:** evita los ciclos `for` sobre filas de un `DataFrame`.
- **Mira tus datos antes de analizarlos:** `info()`, `describe()` e `isna()` al comienzo.
- **Justifica cómo tratas los nulos:** eliminar y rellenar tienen costos distintos.
- **Entre paréntesis cada condición** cuando combines filtros con `&` y `|`.
- **Reasigna el resultado** (`df = df.drop(...)`) o guárdalo en otra variable; las operaciones no modifican el original por sí solas.

### Ejercicios propuestos

1. ¿Cuál fue el **mes** con mayores ventas totales en todo el dataset? *(Pista: `df["fecha"].dt.month` + `groupby`.)*
2. Calcula la venta promedio por **año** y por **ciudad**. *(Pista: `groupby` con una lista de columnas.)*
3. ¿Qué porcentaje de las ventas de cada ciudad cae en el tramo `'Alto'`? *(Pista: `pd.crosstab(..., normalize="index")`.)*
4. Crea una columna `venta_con_iva` (19 %) y compárala con `venta` usando `describe()`.
5. Repite el análisis de nulos usando la **Opción A** (`df_limpio`) y compara los Top 5 de tiendas contra los obtenidos con la Opción B. ¿Cambia el ranking?